# SimpleBCI local training

Run the sections one at a time: record data, train the model, then export ONNX.

**Note:** Synthetic board data is only for checking the software pipeline. It is not real EEG training data.

## Setup (run once)

Install the packages in the active notebook environment. Run this cell once, then continue below.

In [ ]:
%pip install brainflow playsound3 numpy pandas scipy scikit-learn joblib onnx onnxruntime skl2onnx

In [1]:
from pathlib import Path
import subprocess
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / 'record_eeg.py').is_file():
    ROOT = ROOT / 'SimpleBCI.ML'
if not (ROOT / 'record_eeg.py').is_file():
    raise FileNotFoundError('Open this notebook from the SimpleBCI project or SimpleBCI.ML folder.')

RECORDINGS = ROOT / 'motor_imagery_recordings'

def run_script(name, *args):
    result = subprocess.run([sys.executable, str(ROOT / name), *map(str, args)], cwd=ROOT, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError(f'{name} failed with exit code {result.returncode}; see output above.')

print('Project folder:', ROOT)
print('Recording sessions:', len(list(RECORDINGS.glob('*.csv'))) if RECORDINGS.exists() else 0)

Project folder: C:\Users\Riley\Documents\Project Work\SimpleBCI\SimpleBCI.ML
Recording sessions: 0


## 1. Record data

Choose `synthetic` or `muse_s_athena`. The recorder saves the board's actual sampling rate (Synthetic reports 250 Hz; Athena is expected at 256 Hz) and the training script uses that rate. For Athena, BrainFlow's `preset=p1041;low_latency=true` option is set and the four channels are mapped to the training order. Each run creates one session.

In [3]:
BOARD = 'synthetic'  # Use 'muse_s_athena' when the device is available
MAC_ADDRESS = ''     # Optional; leave blank for BrainFlow discovery
ROUNDS = 5
SECONDS_PER_TRIAL = 10
GAP_SECONDS = 2

record_args = ['--board', BOARD, '--rounds', ROUNDS, '--seconds', SECONDS_PER_TRIAL, '--gap', GAP_SECONDS]
if MAC_ADDRESS:
    record_args += ['--mac-address', MAC_ADDRESS]
run_script('record_eeg.py', *record_args)

Traceback (most recent call last):
  File "C:\Users\Riley\Documents\Project Work\SimpleBCI\SimpleBCI.ML\record_eeg.py", line 10, in <module>
    import sounddevice as sd
ModuleNotFoundError: No module named 'sounddevice'



RuntimeError: record_eeg.py failed with exit code 1; see output above.

## 2. Train the model

At least two recording sessions are needed for the session-level train/test split.

In [ ]:
run_script('train_model.py')

## 3. Export ONNX

The ONNX model is saved to `SimpleBCI.Library/Model/Model.onnx`.

In [ ]:
run_script('export_model.py')

## 4. Delete recorded data

This removes recording CSV files and their metadata only. The trained model and ONNX file are kept. Review the list and confirm when prompted.

In [ ]:
files = []
if RECORDINGS.exists():
    files = [p for p in RECORDINGS.iterdir() if p.is_file() and (p.suffix.lower() == '.csv' or p.name.endswith('_metadata.txt'))]

if not files:
    print('No recording files found.')
else:
    for path in files:
        print(path.name)
    if input('Type DELETE RECORDINGS to confirm: ') == 'DELETE RECORDINGS':
        for path in files:
            path.unlink()
        print(f'Deleted {len(files)} files.')
    else:
        print('Cancelled. No files were deleted.')